# Bài 5 trang 29: Gom cụm khối u lành tính và ác tính

**Họ tên:** Lê Nguyễn Anh Du  
**MSSV:** 6451071011

Nguồn: [Breast Cancer Wisconsin (Diagnostic), UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic). Bài làm dùng k-Means để gom dữ liệu thành hai cụm. Nhãn chẩn đoán chỉ được dùng để so sánh sau khi gom cụm. Kết quả là phân tích học tập, không dùng để chẩn đoán bệnh.

In [ ]:
print("Họ tên: Lê Nguyễn Anh Du | MSSV: 6451071011")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, accuracy_score, classification_report, confusion_matrix
)
from scipy.optimize import linear_sum_assignment

HO_TEN = "Lê Nguyễn Anh Du"
MSSV = "6451071011"
def in_thong_tin():
    print(f"Họ tên: {HO_TEN} | MSSV: {MSSV}")

in_thong_tin()
print("Đã nhập các thư viện cần thiết.")

## 1) Đọc dữ liệu từ UCI

Tệp `wdbc.data` có các cột: mã mẫu, nhãn chẩn đoán và 30 đặc trưng số. Nếu đường tải UCI tạm thời không hoạt động, notebook dùng bản sao cùng bộ dữ liệu có sẵn trong scikit-learn.

In [ ]:
in_thong_tin()
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
base_features = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"
]
feature_names = [f"{group}_{feature}" for group in ("mean", "se", "worst") for feature in base_features]
column_names = ["ID", "Diagnosis"] + feature_names
try:
    df = pd.read_csv(DATA_URL, header=None, names=column_names)
    if df.shape[1] != 32 or not set(df["Diagnosis"].dropna().astype(str).str.strip()).issubset({"B", "M"}):
        raise ValueError("Cấu trúc dữ liệu UCI không đúng như mong đợi")
    source_name = "tệp wdbc.data từ UCI"
except Exception as exc:
    bundled = load_breast_cancer()
    df = pd.DataFrame(bundled.data, columns=feature_names)
    df.insert(0, "Diagnosis", np.where(bundled.target == 0, "M", "B"))
    source_name = "bản sao dữ liệu trong scikit-learn"
    print(f"Không đọc được tệp UCI ({type(exc).__name__}); đã dùng bản sao tương ứng.")

print("Nguồn đang dùng:", source_name)
print("Kích thước dữ liệu:", df.shape)
print("5 dòng đầu tiên:")
display(df.head())

## 2) Khảo sát và xử lý dữ liệu

`ID` chỉ là mã mẫu, không tham gia gom cụm. `Diagnosis` là nhãn thật (`B`: lành tính; `M`: ác tính), giữ riêng để đánh giá. Các đặc trưng số được chuyển về dạng số và giá trị thiếu (nếu có) được thay bằng trung vị của cột.

In [ ]:
in_thong_tin()
print("Số giá trị thiếu trước xử lý:", int(df.isna().sum().sum()))
df["Diagnosis"] = df["Diagnosis"].astype("string").str.strip().str.upper()
if df["Diagnosis"].isna().any() or not df["Diagnosis"].isin(["B", "M"]).all():
    raise ValueError("Cột Diagnosis có nhãn thiếu hoặc không hợp lệ.")
X_df = df[feature_names].copy()
for column in feature_names:
    X_df[column] = pd.to_numeric(X_df[column], errors="coerce")
    X_df[column] = X_df[column].fillna(X_df[column].median())
y_true = df["Diagnosis"].to_numpy(dtype=str)
print("Số giá trị thiếu của đặc trưng sau xử lý:", int(X_df.isna().sum().sum()))
print("Số mẫu theo nhãn thật:")
print(pd.Series(y_true).value_counts().to_string())
print("Thống kê mô tả 5 đặc trưng đầu tiên:")
display(X_df.iloc[:, :5].describe().transpose())

## 3) Chuẩn hóa đặc trưng và gom thành hai cụm bằng k-Means

Các đặc trưng có thang đo khác nhau, vì vậy dùng `StandardScaler` trước k-Means. Số cụm đặt bằng 2 vì mục tiêu bài toán là gom thành hai nhóm.

In [ ]:
in_thong_tin()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)
kmeans = KMeans(n_clusters=2, init="k-means++", n_init=20, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)
print("Kích thước ma trận đặc trưng:", X_scaled.shape)
print("Số mẫu mỗi cụm:")
print(pd.Series(cluster_labels).value_counts().sort_index().to_string())
print(f"Inertia: {kmeans.inertia_:.4f}")
print(f"Silhouette coefficient: {silhouette_score(X_scaled, cluster_labels):.4f}")

## 4) Đối chiếu cụm với nhãn lành tính và ác tính

Số cụm `0` và `1` do thuật toán đặt tùy ý. Bảng chéo cho biết số mẫu `B` và `M` trong từng cụm.

In [ ]:
in_thong_tin()
comparison = pd.crosstab(
    pd.Series(cluster_labels, name="Cụm k-Means"),
    pd.Series(y_true, name="Nhãn thật")
).reindex(index=[0, 1], columns=["B", "M"], fill_value=0)
display(comparison)
plt.figure(figsize=(6, 4))
sns.heatmap(comparison, annot=True, fmt="d", cmap="Blues")
plt.title("Số mẫu lành tính và ác tính trong từng cụm")
plt.xlabel("Nhãn thật")
plt.ylabel("Cụm k-Means")
plt.tight_layout()
plt.show()

## 5) Đánh giá mức độ phù hợp giữa cụm và nhãn thật

ARI và NMI so sánh cách chia nhóm mà không phụ thuộc số hiệu cụm. Để tính tỷ lệ trùng nhãn, dùng thuật toán Hungarian ghép hai cụm với `B` và `M` sao cho số mẫu trùng nhãn là lớn nhất. Việc ghép này chỉ phục vụ đánh giá trên dữ liệu hiện có.

In [ ]:
in_thong_tin()
row_index, column_index = linear_sum_assignment(-comparison.to_numpy())
cluster_to_label = {int(cluster): comparison.columns[int(label_index)] for cluster, label_index in zip(row_index, column_index)}
y_mapped = np.array([cluster_to_label[int(cluster)] for cluster in cluster_labels])
print("Ghép cụm với nhãn:", cluster_to_label)
print(f"Adjusted Rand Index (ARI): {adjusted_rand_score(y_true, cluster_labels):.4f}")
print(f"Normalized Mutual Information (NMI): {normalized_mutual_info_score(y_true, cluster_labels):.4f}")
print(f"Tỷ lệ trùng nhãn sau ghép: {accuracy_score(y_true, y_mapped):.4f}")
print("Precision, Recall và F1-score sau khi ghép nhãn:")
print(classification_report(y_true, y_mapped, labels=["B", "M"], target_names=["Lành tính (B)", "Ác tính (M)"], digits=4, zero_division=0))

In [ ]:
in_thong_tin()
cm = confusion_matrix(y_true, y_mapped, labels=["B", "M"])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=["Lành tính", "Ác tính"], yticklabels=["Lành tính", "Ác tính"])
plt.title("Nhãn thật và cụm sau khi ghép")
plt.xlabel("Nhãn gán cho cụm")
plt.ylabel("Nhãn thật")
plt.tight_layout()
plt.show()

## 6) Biểu diễn dữ liệu trên mặt phẳng PCA

PCA chỉ dùng để vẽ hai chiều. K-Means ở trên đã được huấn luyện trên đầy đủ 30 đặc trưng đã chuẩn hóa.

In [ ]:
in_thong_tin()
coordinates = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, color, name in [("B", "tab:blue", "Lành tính"), ("M", "tab:red", "Ác tính")]:
    mask = y_true == label
    axes[0].scatter(coordinates[mask, 0], coordinates[mask, 1], s=22, alpha=0.7, c=color, label=name)
for cluster in [0, 1]:
    mask = cluster_labels == cluster
    axes[1].scatter(coordinates[mask, 0], coordinates[mask, 1], s=22, alpha=0.7, label=f"Cụm {cluster} → {cluster_to_label[cluster]}")
axes[0].set_title("Màu theo nhãn chẩn đoán thật")
axes[1].set_title("Màu theo cụm k-Means")
for ax in axes:
    ax.set_xlabel("Thành phần PCA 1")
    ax.set_ylabel("Thành phần PCA 2")
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 7) Nhận xét kết quả

Quan sát bảng chéo và ma trận nhầm lẫn để xem số mẫu lành tính hoặc ác tính được xếp vào cụm còn lại. ARI, NMI và Silhouette đánh giá những khía cạnh khác nhau của việc gom cụm; tỷ lệ trùng nhãn sau ghép cho biết mức độ tương ứng với hai nhãn có sẵn.

In [ ]:
in_thong_tin()
wrong = int(np.sum(y_true != y_mapped))
print(f"Số mẫu được gom vào cụm tương ứng với nhãn thật: {len(y_true) - wrong}/{len(y_true)}")
print(f"Số mẫu có nhãn thật khác nhãn gán cho cụm: {wrong}")
for cluster in [0, 1]:
    row = comparison.loc[cluster]
    print(f"Cụm {cluster}: {int(row['B'])} mẫu lành tính, {int(row['M'])} mẫu ác tính.")